In [157]:
import pandas as pd
import numpy as np

ds = pd.read_csv('../../../data/week4_korea_cli.csv')
ds

,observation_date,KORLOLITOAASTSAM
0,1990-01-01,100.461861
1,1990-02-01,100.462284
2,1990-03-01,100.536374
3,1990-04-01,100.613811
4,1990-05-01,100.646672
...,...,...
433,2026-02-01,101.597894
434,2026-03-01,102.001885
435,2026-04-01,102.363604
436,2026-05-01,102.657914


In [158]:
ds.observation_date = pd.to_datetime(ds.observation_date)
ds = ds.sort_values('observation_date')

In [159]:
ds['target_next_month'] = ds['KORLOLITOAASTSAM'].shift(-1).copy()
ds['baseline'] = ds['KORLOLITOAASTSAM'].copy()
ds['cli_lag1'] = ds['KORLOLITOAASTSAM'].shift(1).copy()
ds['cli_rolling3'] = ds['KORLOLITOAASTSAM'].rolling(window=3).mean()
ds['cli_diff1'] = ds['KORLOLITOAASTSAM'].diff(1)

In [160]:
month = ds['observation_date'].dt.month

ds['month_sin'] = np.sin(2 * np.pi * month / 12)
ds['month_cos'] = np.cos(2 * np.pi * month / 12)

In [161]:
set_A = ['cli_lag1', 'cli_rolling3']
set_B = ['cli_lag1', 'cli_rolling3', 'cli_diff1']
set_C = ['cli_lag1', 'cli_rolling3', 'cli_diff1', 'month_sin', 'month_cos']

In [162]:
ds

,observation_date,KORLOLITOAASTSAM,target_next_month,baseline,cli_lag1,cli_rolling3,cli_diff1,month_sin,month_cos
0,1990-01-01,100.461861,100.462284,100.461861,NaN,NaN,NaN,5.000000e-01,8.660254e-01
1,1990-02-01,100.462284,100.536374,100.462284,100.461861,NaN,0.000423,8.660254e-01,5.000000e-01
2,1990-03-01,100.536374,100.613811,100.536374,100.462284,100.486840,0.074089,1.000000e+00,6.123234e-17
3,1990-04-01,100.613811,100.646672,100.613811,100.536374,100.537490,0.077438,8.660254e-01,-5.000000e-01
4,1990-05-01,100.646672,100.610137,100.646672,100.613811,100.598952,0.032861,5.000000e-01,-8.660254e-01
...,...,...,...,...,...,...,...,...,...
433,2026-02-01,101.597894,102.001885,101.597894,101.190196,101.200525,0.407698,8.660254e-01,5.000000e-01
434,2026-03-01,102.001885,102.363604,102.001885,101.597894,101.596658,0.403991,1.000000e+00,6.123234e-17
435,2026-04-01,102.363604,102.657914,102.363604,102.001885,101.987795,0.361719,8.660254e-01,-5.000000e-01
436,2026-05-01,102.657914,102.869808,102.657914,102.363604,102.341134,0.294310,5.000000e-01,-8.660254e-01


In [163]:
ds = ds.dropna(subset=['cli_lag1', 'cli_rolling3', 'cli_diff1', 'KORLOLITOAASTSAM', 'target_next_month'])
ds

,observation_date,KORLOLITOAASTSAM,target_next_month,baseline,cli_lag1,cli_rolling3,cli_diff1,month_sin,month_cos
2,1990-03-01,100.536374,100.613811,100.536374,100.462284,100.486840,0.074089,1.000000e+00,6.123234e-17
3,1990-04-01,100.613811,100.646672,100.613811,100.536374,100.537490,0.077438,8.660254e-01,-5.000000e-01
4,1990-05-01,100.646672,100.610137,100.646672,100.613811,100.598952,0.032861,5.000000e-01,-8.660254e-01
5,1990-06-01,100.610137,100.477904,100.610137,100.646672,100.623540,-0.036535,1.224647e-16,-1.000000e+00
6,1990-07-01,100.477904,100.254061,100.477904,100.610137,100.578238,-0.132233,-5.000000e-01,-8.660254e-01
...,...,...,...,...,...,...,...,...,...
432,2026-01-01,101.190196,101.597894,101.190196,100.813483,100.830936,0.376713,5.000000e-01,8.660254e-01
433,2026-02-01,101.597894,102.001885,101.597894,101.190196,101.200525,0.407698,8.660254e-01,5.000000e-01
434,2026-03-01,102.001885,102.363604,102.001885,101.597894,101.596658,0.403991,1.000000e+00,6.123234e-17
435,2026-04-01,102.363604,102.657914,102.363604,102.001885,101.987795,0.361719,8.660254e-01,-5.000000e-01


In [164]:
ds.columns

Index(['observation_date', 'KORLOLITOAASTSAM', 'target_next_month', 'baseline',
       'cli_lag1', 'cli_rolling3', 'cli_diff1', 'month_sin', 'month_cos'],
      dtype='str')

In [165]:
ds.observation_date.min(), ds.observation_date.max()

(Timestamp('1990-03-01 00:00:00'), Timestamp('2026-05-01 00:00:00'))

In [173]:
train = ds[(ds.observation_date >= pd.to_datetime('2000-01-01')) & (ds.observation_date <= pd.to_datetime('2014-12-01'))].copy()
validation = ds[(ds.observation_date >= pd.to_datetime('2015-01-01')) & (ds.observation_date <= pd.to_datetime('2019-12-01'))].copy()

len(train), len(validation)

(180, 60)

In [167]:
from sklearn.linear_model import LinearRegression

model_A = LinearRegression()
model_B = LinearRegression()
model_C = LinearRegression()

In [168]:
X_train_A = train[set_A].copy()
X_train_B = train[set_B].copy()
X_train_C = train[set_C].copy()

y_train = train['target_next_month']

In [169]:
model_A.fit(X_train_A, y_train)
model_B.fit(X_train_B, y_train)
model_C.fit(X_train_C, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](5,)","[-1.57, 2.56, 1.92,-0. , 0.01]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](5,)","['cli_lag1','cli_rolling3','cli_diff1','month_sin','month_cos']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,0.6339
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,5
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(5)


In [ ]:
validation['model_predict_A'] = model_A.predict(validation[set_A])
validation['model_predict_B'] = model_B.predict(validation[set_B])
validation['model_predict_C'] = model_C.predict(validation[set_C])

In [ ]:
y_answer = validation.target_next_month.copy()

result = pd.DataFrame(index=['base', 'A', 'B', 'C'])

result.loc['base', 'model'] = 'baseline'
result.loc['base', 'feature_count'] = 1
result.loc['base', 'validation_rows'] = len(validation)
result.loc['base', 'MAE'] = (y_answer - validation.baseline).abs().mean()
result.loc['base', 'RMSE'] = np.sqrt(((y_answer - validation.baseline)**2).mean())

result.loc['A', 'model'] = 'model A'
result.loc['A', 'feature_count'] = len(set_A)
result.loc['A', 'validation_rows'] = len(validation)
result.loc['A', 'MAE'] = (y_answer - validation.model_predict_A).abs().mean()
result.loc['A', 'RMSE'] = np.sqrt(((y_answer - validation.model_predict_A)**2).mean())


result.loc['B', 'model'] = 'model B'
result.loc['B', 'feature_count'] = len(set_B)
result.loc['B', 'validation_rows'] = len(validation)
result.loc['B', 'MAE'] = (y_answer - validation.model_predict_B).abs().mean()
result.loc['B', 'RMSE'] = np.sqrt(((y_answer - validation.model_predict_B)**2).mean())


result.loc['C', 'model'] = 'model C'
result.loc['C', 'feature_count'] = len(set_C)
result.loc['C', 'validation_rows'] = len(validation)
result.loc['C', 'MAE'] = (y_answer - validation.model_predict_C).abs().mean()
result.loc['C', 'RMSE'] = np.sqrt(((y_answer - validation.model_predict_C)**2).mean())

In [172]:
result

,model,feature_count,validation_rows,MAE,RMSE
base,baseline,1.0,60.0,0.079752,0.093522
A,model A,2.0,60.0,0.177627,0.211966
B,model B,3.0,60.0,0.010768,0.013730
C,model C,5.0,60.0,0.012138,0.015409
